# 02 — Il dataset definitivo

Dal dataset grezzo (scraping + filtri base) al dataset su cui classifichiamo. Il disegno è uno solo:

1. **testate con almeno 20 articoli** (un voto di maggioranza su pochi articoli non è affidabile);
2. **pulizia degli articoli**, cieca alle etichette: una sola copia per articolo gemello, via le copie tra
   gruppi diversi, via i paragrafi condivisi da più testate, via gli articoli rimasti con ≤ 100 parole;
3. **pagine del sito**: si tengono solo i tipi informativi (about, ownership, corrections, staff, ethics,
   contact), si toglie il boilerplate condiviso, e si **tengono solo le testate che hanno ancora una pagina
   about utile**;
4. **fold** per gruppo e per testata, stratificati per livello.

I gruppi di testate parenti vengono dal notebook 01.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from comune import carica_dataset, normalizza, DERIVED, RESULTS, SEED, LIVELLI

VERS = "../Dataset/versioni/"; import os; os.makedirs(VERS, exist_ok=True)
PAGINE = "../Dataset/NuovoScraping/dataset_pagine_sito.csv"
INFORMATIVE = ["about", "ownership", "corrections", "staff", "ethics", "contact"]

df = carica_dataset()
gruppo = pd.read_csv(RESULTS + "01_gruppi_testate.csv", index_col="domain").gruppo
df["gruppo"] = df.domain.map(gruppo)
aid = df.article_id.values
coppie = pd.read_csv(DERIVED + "01_coppie_quasi_uguali.csv.gz")
coppie["a"], coppie["b"] = aid[coppie.articolo_a], aid[coppie.articolo_b]
print(len(df), "articoli,", df.domain.nunique(), "testate")

54532 articoli, 354 testate


## 1. Solo testate con almeno 20 articoli

In [2]:
per_t = df.domain.value_counts()
df = df[df.domain.map(per_t) >= 20].reset_index(drop=True)
print("dopo il minimo di 20 articoli:", len(df), "articoli,", df.domain.nunique(), "testate")

dopo il minimo di 20 articoli: 54276 articoli, 317 testate


## 2. Pulizia degli articoli (quattro regole)

In [3]:
tolti = {}
# regola 1: una sola copia per articolo gemello (Jaccard >= 0.9)
G = nx.Graph(); G.add_edges_from(coppie[(coppie.jaccard >= 0.9) & coppie.a.isin(df.article_id) & coppie.b.isin(df.article_id)][["a", "b"]].itertuples(index=False))
rng = np.random.default_rng(SEED)
for fam in nx.connected_components(G):
    tieni = rng.choice(sorted(fam))
    for x in fam:
        if x != tieni: tolti[x] = "copia ripresa"
# regola 2: via le copie tra gruppi diversi
rimasti = set(df.article_id) - set(tolti)
c = coppie[coppie.a.isin(rimasti) & coppie.b.isin(rimasti)]
tra = c[c.testata_a.map(gruppo) != c.testata_b.map(gruppo)]
for x in set(tra.a) | set(tra.b): tolti[x] = "copia tra gruppi"
# regola 3: via i paragrafi condivisi da >=2 testate
vivi = df[~df.article_id.isin(tolti)]
par = vivi[["article_id", "domain", "text"]].assign(p=vivi.text.astype(str).str.split("\n")).explode("p")
par["n"] = par.p.map(normalizza); par = par[par.n.str.split().str.len() >= 5]
par["cond"] = par.n.map(par.groupby("n").domain.nunique()) >= 2
testo = par[~par.cond].groupby("article_id").p.agg("\n".join)
df["testo_pulito"] = testo.reindex(df.article_id).fillna("").values
# regola 4: via gli articoli rimasti con <= 100 parole
df["parole"] = df.testo_pulito.str.split().str.len()
for x in df[~df.article_id.isin(tolti) & (df.parole <= 100)].article_id: tolti[x] = "troppo corto"
df["motivo"] = df.article_id.map(tolti).fillna("tenuto")
print(df.motivo.value_counts().to_string())
puliti = df[df.motivo == "tenuto"].copy()
print("\narticoli dopo la pulizia:", len(puliti), "— testate:", puliti.domain.nunique())

motivo
tenuto              48242
copia ripresa        4656
troppo corto         1314
copia tra gruppi       64

articoli dopo la pulizia: 48242 — testate: 317


## 3. Pagine del sito: solo informative, pulite, e richieste

In [4]:
pag = pd.read_csv(PAGINE)
pag = pag[pag.dominio.isin(puliti.domain.unique()) & pag.tipo.isin(INFORMATIVE)].copy()
parp = pag.assign(p=pag.testo.astype(str).str.split("\n")).explode("p")
parp["n"] = parp.p.map(normalizza); parp = parp[parp.n.str.split().str.len() >= 5]
parp["cond"] = parp.n.map(parp.groupby("n").dominio.nunique()) >= 2
testo_pag = parp[~parp.cond].groupby("dominio").p.agg("\n".join)
testo_pag = testo_pag[testo_pag.str.split().str.len() >= 10]           # pagina about utile non vuota
tipi = pag[pag.dominio.isin(testo_pag.index)].groupby("dominio").tipo.apply(lambda s: " ".join(sorted(set(s))))
print(f"paragrafi di pagine condivisi tolti: {parp.cond.mean():.0%}")
print("testate con pagina about utile dopo la pulizia:", testo_pag.index.nunique())

paragrafi di pagine condivisi tolti: 20%
testate con pagina about utile dopo la pulizia: 272


## 4. Dataset definitivo: solo le testate con articoli E pagina about

In [5]:
tenute = sorted(testo_pag.index)
puliti = puliti[puliti.domain.isin(tenute)].reset_index(drop=True)
testate = puliti.groupby("domain").agg(label=("label", "first"), gruppo=("gruppo", "first")).reset_index().sort_values("domain").reset_index(drop=True)
sg = StratifiedGroupKFold(5, shuffle=True, random_state=SEED); testate["fold_gruppo"] = 0
for i, (_, idx) in enumerate(sg.split(testate.index, testate.label, groups=testate.gruppo), 1): testate.loc[idx, "fold_gruppo"] = i
sk = StratifiedKFold(5, shuffle=True, random_state=SEED); testate["fold_testata"] = 0
for i, (_, idx) in enumerate(sk.split(testate.index, testate.label), 1): testate.loc[idx, "fold_testata"] = i

comp = testate.groupby("label").size().reindex(range(5), fill_value=0); comp.index = LIVELLI
comp2 = pd.DataFrame({"testate": comp, "articoli": puliti.groupby("label").size().values})
comp2.loc["totale"] = comp2.sum()
print("DATASET DEFINITIVO:"); print(comp2.to_string())
display(pd.crosstab(testate.fold_gruppo, testate.label.map(lambda k: LIVELLI[k]), margins=True, margins_name="tot"))

DATASET DEFINITIVO:
        testate  articoli
l1           42      7291
l2           46      8145
l3           75     11074
l4           81     13262
l5           28      5163
totale      272     44935


label,l1,l2,l3,l4,l5,tot
fold_gruppo,,,,,,
1,8,9,31,16,5,69
2,9,9,11,16,6,51
3,9,9,11,17,5,51
4,8,9,11,16,6,50
5,8,10,11,16,6,51
tot,42,46,75,81,28,272


## 5. Salvataggio

In [6]:
puliti = puliti.rename(columns={"testo_pulito": "testo"})
puliti[["article_id", "domain", "gruppo", "label", "testo"]].to_csv(VERS + "dataset_finale.csv", index=False)
testate.to_csv(VERS + "fold_finale.csv", index=False)
pagine = testate.set_index("domain").join(testo_pag.rename("testo_pagine")).join(tipi.rename("tipi_pagine"))
pagine.to_csv(VERS + "pagine_finale.csv")
print("Salvato dataset_finale, fold_finale, pagine_finale:", len(puliti), "articoli,", len(testate), "testate")

Salvato dataset_finale, fold_finale, pagine_finale: 44935 articoli, 272 testate
